# CineMatch AI: content-based movie recommendation\nThis notebook documents the same TF-IDF and cosine-similarity pipeline used by the FastAPI application.

## 1. Load the project data\nThe original raw TMDB CSV is not in this repository. We intentionally use the preserved processed `ew.pickle` artifact and never alter it.

In [ ]:
from pathlib import Path\nimport pickle\nimport pandas as pd\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\n\nROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()\nwith (ROOT / 'models' / 'ew.pickle').open('rb') as f:\n    raw_movies = pickle.load(f)\nraw_movies.head()

## 2. Inspect shape and columns

In [ ]:
print('Shape:', raw_movies.shape)\nprint('Columns:', raw_movies.columns.tolist())\nraw_movies.sample(min(5, len(raw_movies)), random_state=42)

## 3. Missing values and duplicates\nWe analyze before changing anything, then make a clean in-memory training copy.

In [ ]:
display(raw_movies.isna().sum().sort_values(ascending=False))\nprint('Duplicate rows:', raw_movies.duplicated().sum())\nprint('Duplicate titles:', raw_movies['title'].duplicated().sum())

## 4. Cleaning and feature engineering\n`tags` is the original combined text feature: overview, genres, keywords, leading cast, and director. Empty titles are removed and duplicate titles retain their first record.

In [ ]:
movies = raw_movies.copy()\nmovies['title'] = movies['title'].fillna('').astype(str).str.strip()\nmovies['features'] = movies.get('tags', '').fillna('').astype(str)\nmovies = movies[movies['title'].ne('')].drop_duplicates('title').reset_index(drop=True)\nmovies[['title', 'features']].head()

## 5. TF-IDF vectorization and cosine similarity\nTF-IDF downweights very common words. Cosine similarity compares normalized vectors, making the score suitable for movie descriptions of different lengths.

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=12_000)\ntfidf_matrix = vectorizer.fit_transform(movies['features'])\nprint(tfidf_matrix.shape)\n\ndef recommend(title, n=6):\n    matches = movies.index[movies['title'].str.casefold() == title.strip().casefold()].tolist()\n    if not matches:\n        raise KeyError(f'Unknown movie title: {title}')\n    index = matches[0]\n    scores = cosine_similarity(tfidf_matrix[index], tfidf_matrix).ravel()\n    nearest = [i for i in scores.argsort()[::-1] if i != index][:n]\n    return movies.loc[nearest, ['title']].assign(similarity=scores[nearest])

## 6. Example recommendations and basic validation

In [ ]:
example_title = movies['title'].iloc[0]\nrecommendations = recommend(example_title)\ndisplay(recommendations)\nassert example_title not in recommendations['title'].tolist()\nassert len(recommendations) <= 6\ntry:\n    recommend('A definitely unknown movie title')\nexcept KeyError:\n    print('Unknown-title handling works.')

## 7. Conclusion\nThe production API builds the vectorizer once at application startup and uses this same recommendation function concept for each request. This avoids rebuilding the expensive document matrix for every visitor.